my first notebook


In [1]:
import pandas as pd
df = pd.read_csv("../data/payments.csv")

df.head()


,payment_id,customer_id,amount,payment_method,payment_status,failure_reason,failed_attempts,previous_successful_payments,days_since_last_payment,checkout_abandoned,recovered,recovery_action,recovered_amount
0,P00001,C1103,409.59,UPI,success,none,0,4,162,no,no,none,0.0
1,P00002,C1180,4086.34,UPI,failed,bank_decline,3,1,75,no,no,stop,0.0
2,P00003,C1093,1358.44,Wallet,success,none,0,3,95,no,no,none,0.0
3,P00004,C1015,3745.69,Card,success,none,0,4,23,no,no,none,0.0
4,P00005,C1107,447.73,UPI,success,none,0,6,124,no,no,none,0.0


In [2]:
df.shape 
df.info()
df.describe()


<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 13 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   payment_id                    1000 non-null   str    
 1   customer_id                   1000 non-null   str    
 2   amount                        1000 non-null   float64
 3   payment_method                1000 non-null   str    
 4   payment_status                1000 non-null   str    
 5   failure_reason                1000 non-null   str    
 6   failed_attempts               1000 non-null   int64  
 7   previous_successful_payments  1000 non-null   int64  
 8   days_since_last_payment       1000 non-null   int64  
 9   checkout_abandoned            1000 non-null   str    
 10  recovered                     1000 non-null   str    
 11  recovery_action               1000 non-null   str    
 12  recovered_amount              1000 non-null   float64
dtypes: float64(2), 

,amount,failed_attempts,previous_successful_payments,days_since_last_payment,recovered_amount
count,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000
mean,2020.104010,1.186000,4.906000,88.520000,411.071780
std,1697.755124,1.715327,2.060439,53.065257,1133.978791
min,168.650000,0.000000,0.000000,1.000000,0.000000
25%,926.342500,0.000000,3.000000,42.000000,0.000000
50%,1521.535000,0.000000,5.000000,86.000000,0.000000
75%,2462.952500,2.000000,6.000000,135.000000,0.000000
max,14900.660000,5.000000,14.000000,180.000000,10660.350000


In [3]:
df["payment_status"].value_counts()

payment_status
success    611
failed     389
Name: count, dtype: int64

In [4]:
failed_payments = df[df["payment_status"] == "failed"]

revenue_at_risk = failed_payments["amount"].sum()

print("Revenue at Risk:", revenue_at_risk)

Revenue at Risk: 822299.2


In [5]:
recovered_revenue = df["recovered_amount"].sum()

print("Recovered Revenue:", recovered_revenue)

Recovered Revenue: 411071.78


In [6]:
recovery_rate = (recovered_revenue / revenue_at_risk) * 100

print("Recovery Rate:", round(recovery_rate, 2), "%")

Recovery Rate: 49.99 %


In [7]:
failed_payments["failure_reason"].value_counts()

failure_reason
bank_decline             113
insufficient_funds        94
technical_timeout         81
authentication_failed     51
network_error             50
Name: count, dtype: int64

In [8]:
df.shape

df["payment_status"].value_counts()

print("Revenue At Risk:", revenue_at_risk)
print("Recovered Revenue:", recovered_revenue)
print("Recovery Rate:", round(recovery_rate, 2), "%")

Revenue At Risk: 822299.2
Recovered Revenue: 411071.78
Recovery Rate: 49.99 %


In [9]:
failed_payments["failure_reason"].value_counts()
failed_payments.groupby("failure_reason")["amount"].sum().sort_values(ascending=False)

failure_reason
bank_decline             251924.22
technical_timeout        171639.07
insufficient_funds       167648.87
authentication_failed    121620.78
network_error            109466.26
Name: amount, dtype: float64

In [10]:
failed_payments.groupby("payment_method")["amount"].sum().sort_values(ascending=False)
failed_payments.groupby("payment_method").size().sort_values(ascending=False)

payment_method
UPI           202
Card          120
Netbanking     47
Wallet         20
dtype: int64

In [11]:
failed_payments.groupby("payment_method")["amount"].sum().sort_values(ascending=False)

payment_method
UPI           409424.82
Card          278849.87
Netbanking     90687.61
Wallet         43336.90
Name: amount, dtype: float64

In [12]:
failed_payments.groupby("failure_reason").agg(
    failed_payments=("payment_id", "count"),
    recovered_payments=("recovered", lambda x: (x == "yes").sum()),
    revenue_at_risk=("amount", "sum"),
    recovered_revenue=("recovered_amount", "sum")
)

,failed_payments,recovered_payments,revenue_at_risk,recovered_revenue
failure_reason,,,,
authentication_failed,51,22,121620.78,60923.69
bank_decline,113,55,251924.22,122404.49
insufficient_funds,94,38,167648.87,69912.98
network_error,50,32,109466.26,76859.96
technical_timeout,81,39,171639.07,80970.66


In [13]:
failed_payments.groupby("payment_method").agg(
    failed_payments=("payment_id", "count"),
    recovered_payments=("recovered", lambda x: (x == "yes").sum()),
    revenue_at_risk=("amount", "sum"),
    recovered_revenue=("recovered_amount", "sum")
)

,failed_payments,recovered_payments,revenue_at_risk,recovered_revenue
payment_method,,,,
Card,120,48,278849.87,116714.70
Netbanking,47,21,90687.61,46487.58
UPI,202,109,409424.82,230175.01
Wallet,20,8,43336.90,17694.49


In [14]:
failed_payments.groupby("failed_attempts").agg(
    failed_payments=("payment_id", "count"),
    recovered_payments=("recovered", lambda x: (x == "yes").sum()),
    revenue_at_risk=("amount", "sum"),
    recovered_revenue=("recovered_amount", "sum")
)

,failed_payments,recovered_payments,revenue_at_risk,recovered_revenue
failed_attempts,,,,
1,70,42,175673.62,100810.45
2,72,48,135215.46,99688.19
3,90,35,200047.17,83391.30
4,83,35,149612.74,70015.35
5,74,26,161750.21,57166.49


In [15]:
failed_payments.groupby("previous_successful_payments").agg(
    failed_payments=("payment_id", "count"),
    recovered_payments=("recovered", lambda x: (x == "yes").sum()),
    revenue_at_risk=("amount", "sum"),
    recovered_revenue=("recovered_amount", "sum")
)

,failed_payments,recovered_payments,revenue_at_risk,recovered_revenue
previous_successful_payments,,,,
0,4,1,4903.87,716.04
1,12,6,26901.72,12988.58
2,28,13,62691.93,25786.64
3,51,14,116733.11,29410.63
4,70,22,126142.45,48652.98
5,79,47,169542.00,104145.85
6,62,35,132171.42,81272.05
7,40,26,97127.24,63051.61
8,27,13,58348.37,26461.72


feature 


In [16]:
ml_data = failed_payments[
    [
        "amount",
        "payment_method",
        "failure_reason",
        "failed_attempts",
        "previous_successful_payments",
        "days_since_last_payment",
        "checkout_abandoned",
        "recovered"
    ]
].copy()

ml_data.head()

,amount,payment_method,failure_reason,failed_attempts,previous_successful_payments,days_since_last_payment,checkout_abandoned,recovered
1,4086.34,UPI,bank_decline,3,1,75,no,no
9,2361.70,UPI,technical_timeout,4,2,169,no,yes
12,1620.07,Card,insufficient_funds,4,4,109,no,no
15,636.84,UPI,network_error,4,8,36,no,yes
19,844.41,Card,authentication_failed,5,2,154,yes,no


In [17]:
%whos DataFrame


Variable          Type         Data/Info
----------------------------------------
df                DataFrame    Shape: (1000, 13)
failed_payments   DataFrame    Shape: (389, 13)
ml_data           DataFrame    Shape: (389, 8)


In [18]:
failed_payments = df[df["payment_status"] == "failed"].copy()
failed_payments["recovered"].value_counts(dropna=False)

recovered
no     203
yes    186
Name: count, dtype: int64

In [19]:
ml_data = failed_payments[
    [
        "amount",
        "payment_method",
        "failure_reason",
        "failed_attempts",
        "previous_successful_payments",
        "days_since_last_payment",
        "checkout_abandoned",
        "recovered"
    ]
].copy()
ml_data.head()

,amount,payment_method,failure_reason,failed_attempts,previous_successful_payments,days_since_last_payment,checkout_abandoned,recovered
1,4086.34,UPI,bank_decline,3,1,75,no,no
9,2361.70,UPI,technical_timeout,4,2,169,no,yes
12,1620.07,Card,insufficient_funds,4,4,109,no,no
15,636.84,UPI,network_error,4,8,36,no,yes
19,844.41,Card,authentication_failed,5,2,154,yes,no


In [20]:
ml_data["recovered"] = ml_data["recovered"].map({
    "no": 0,
    "yes": 1
})
ml_data["recovered"].value_counts()

recovered
0    203
1    186
Name: count, dtype: int64

In [21]:
ml_data.columns.tolist()


['amount',
 'payment_method',
 'failure_reason',
 'failed_attempts',
 'previous_successful_payments',
 'days_since_last_payment',
 'checkout_abandoned',
 'recovered']

In [22]:
X = ml_data.drop("recovered", axis=1)
y = ml_data["recovered"]
X.head()
y.head()


1     0
9     1
12    0
15    1
19    0
Name: recovered, dtype: int64

In [23]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (311, 7)
X_test: (78, 7)
y_train: (311,)
y_test: (78,)


In [24]:
ml_data.columns.tolist()

['amount',
 'payment_method',
 'failure_reason',
 'failed_attempts',
 'previous_successful_payments',
 'days_since_last_payment',
 'checkout_abandoned',
 'recovered']

In [25]:
ml_data = pd.get_dummies(
    ml_data,
    columns=["payment_method", "failure_reason", "checkout_abandoned"],
    drop_first=True,
    dtype=int
)
ml_data.columns.tolist()

['amount',
 'failed_attempts',
 'previous_successful_payments',
 'days_since_last_payment',
 'recovered',
 'payment_method_Netbanking',
 'payment_method_UPI',
 'payment_method_Wallet',
 'failure_reason_bank_decline',
 'failure_reason_insufficient_funds',
 'failure_reason_network_error',
 'failure_reason_technical_timeout',
 'checkout_abandoned_yes']

In [26]:
X = ml_data.drop("recovered", axis=1)
y = ml_data["recovered"]
X.columns.tolist()

['amount',
 'failed_attempts',
 'previous_successful_payments',
 'days_since_last_payment',
 'payment_method_Netbanking',
 'payment_method_UPI',
 'payment_method_Wallet',
 'failure_reason_bank_decline',
 'failure_reason_insufficient_funds',
 'failure_reason_network_error',
 'failure_reason_technical_timeout',
 'checkout_abandoned_yes']

In [27]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)

X_train: (311, 12)
X_test: (78, 12)


In [28]:
from sklearn.linear_model import LogisticRegression
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(y_test, y_pred)

print("Accuracy:", accuracy)

Accuracy: 0.6025641025641025


c:\Users\DELL\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [29]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = LogisticRegression(max_iter=1000)

model.fit(X_train_scaled, y_train)

y_pred = model.predict(X_test_scaled)

accuracy = accuracy_score(y_test, y_pred)

print("Accuracy:", accuracy)


Accuracy: 0.5897435897435898


In [30]:
print(y_test.unique())
print(y_pred[:10])

[0 1]
[0 0 1 1 1 0 1 0 0 0]


In [31]:
print(ml_data["recovered"].unique())
print(ml_data["recovered"].isna().sum())

[0 1]
0


In [32]:
failed_payments = df[df["payment_status"] == "failed"].copy()

ml_data = failed_payments[
    [
        "amount",
        "payment_method",
        "failure_reason",
        "failed_attempts",
        "previous_successful_payments",
        "days_since_last_payment",
        "checkout_abandoned",
        "recovered"
    ]
].copy()

ml_data["recovered"] = ml_data["recovered"].map({
    "no": 0,
    "yes": 1
})

print(ml_data["recovered"].value_counts(dropna=False))

recovered
0    203
1    186
Name: count, dtype: int64


In [33]:
ml_data = pd.get_dummies(
    ml_data,
    columns=["payment_method", "failure_reason", "checkout_abandoned"],
    drop_first=True,
    dtype=int
)

In [34]:
print(ml_data.columns.tolist())

['amount', 'failed_attempts', 'previous_successful_payments', 'days_since_last_payment', 'recovered', 'payment_method_Netbanking', 'payment_method_UPI', 'payment_method_Wallet', 'failure_reason_bank_decline', 'failure_reason_insufficient_funds', 'failure_reason_network_error', 'failure_reason_technical_timeout', 'checkout_abandoned_yes']


In [35]:
X = ml_data.drop("recovered", axis=1)
y = ml_data["recovered"]

In [36]:
print("X columns:", X.columns.tolist())
print("y values:", y.unique())
print("NaN in y:", y.isna().sum())

X columns: ['amount', 'failed_attempts', 'previous_successful_payments', 'days_since_last_payment', 'payment_method_Netbanking', 'payment_method_UPI', 'payment_method_Wallet', 'failure_reason_bank_decline', 'failure_reason_insufficient_funds', 'failure_reason_network_error', 'failure_reason_technical_timeout', 'checkout_abandoned_yes']
y values: [0 1]
NaN in y: 0


In [37]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (311, 12)
X_test: (78, 12)
y_train: (311,)
y_test: (78,)


In [38]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [39]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000)

model.fit(X_train_scaled, y_train)

y_pred = model.predict(X_test_scaled)

In [40]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Accuracy: 0.5897435897435898
Precision: 0.5641025641025641
Recall: 0.5945945945945946
F1 Score: 0.5789473684210527

Confusion Matrix:
[[24 17]
 [15 22]]

Classification Report:
              precision    recall  f1-score   support

           0       0.62      0.59      0.60        41
           1       0.56      0.59      0.58        37

    accuracy                           0.59        78
   macro avg       0.59      0.59      0.59        78
weighted avg       0.59      0.59      0.59        78



In [41]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=200,
    random_state=42
)

rf_model.fit(X_train, y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",200
,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",42
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstr

In [42]:
rf_pred = rf_model.predict(X_test)

In [43]:
print("Accuracy:", accuracy_score(y_test, rf_pred))
print("Precision:", precision_score(y_test, rf_pred))
print("Recall:", recall_score(y_test, rf_pred))
print("F1 Score:", f1_score(y_test, rf_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, rf_pred))

Accuracy: 0.6282051282051282
Precision: 0.6052631578947368
Recall: 0.6216216216216216
F1 Score: 0.6133333333333333

Confusion Matrix:
[[26 15]
 [14 23]]


In [44]:
rf_prob = rf_model.predict_proba(X_test)

print(rf_prob[:10])

[[0.735 0.265]
 [0.595 0.405]
 [0.215 0.785]
 [0.375 0.625]
 [0.255 0.745]
 [0.655 0.345]
 [0.67  0.33 ]
 [0.57  0.43 ]
 [0.27  0.73 ]
 [0.825 0.175]]


In [45]:
recovery_probability = rf_model.predict_proba(X_test)[:, 1]

print(recovery_probability[:10])

[0.265 0.405 0.785 0.625 0.745 0.345 0.33  0.43  0.73  0.175]


In [46]:
results = X_test.copy()

results["actual_recovered"] = y_test
results["recovery_probability"] = recovery_probability
results["predicted_recovered"] = rf_pred


In [47]:

results.head(10)

,amount,failed_attempts,previous_successful_payments,days_since_last_payment,payment_method_Netbanking,payment_method_UPI,payment_method_Wallet,failure_reason_bank_decline,failure_reason_insufficient_funds,failure_reason_network_error,failure_reason_technical_timeout,checkout_abandoned_yes,actual_recovered,recovery_probability,predicted_recovered
790,708.49,3,4,166,0,1,0,1,0,0,0,0,0,0.265,0
31,2476.34,5,7,43,0,1,0,1,0,0,0,0,1,0.405,0
519,1824.87,3,6,6,0,1,0,0,1,0,0,0,1,0.785,1
814,582.37,2,8,31,0,1,0,0,0,0,1,0,1,0.625,1
147,4364.14,2,7,55,0,1,0,1,0,0,0,0,1,0.745,1
928,1553.64,3,6,125,0,1,0,0,1,0,0,0,1,0.345,0
580,1054.45,1,4,96,0,1,0,1,0,0,0,0,0,0.330,0
347,1509.55,3,6,89,1,0,0,1,0,0,0,0,0,0.430,0
382,1399.93,5,5,7,0,1,0,1,0,0,0,1,1,0.730,1
462,978.32,3,1,43,0,0,0,0,0,0,1,0,1,0.175,0


In [48]:
def recovery_decision(probability):
    if probability >= 0.70:
        return "high"
    elif probability >= 0.40:
        return "medium"
    else:
        return "low"

In [49]:
results["risk_level"] = results["recovery_probability"].apply(
    recovery_decision
)

In [50]:
results[
    [
        "recovery_probability",
        "risk_level"
    ]
].head(10)

,recovery_probability,risk_level
790,0.265,low
31,0.405,medium
519,0.785,high
814,0.625,medium
147,0.745,high
928,0.345,low
580,0.330,low
347,0.430,medium
382,0.730,high
462,0.175,low


In [51]:
results.columns.tolist()

['amount',
 'failed_attempts',
 'previous_successful_payments',
 'days_since_last_payment',
 'payment_method_Netbanking',
 'payment_method_UPI',
 'payment_method_Wallet',
 'failure_reason_bank_decline',
 'failure_reason_insufficient_funds',
 'failure_reason_network_error',
 'failure_reason_technical_timeout',
 'checkout_abandoned_yes',
 'actual_recovered',
 'recovery_probability',
 'predicted_recovered',
 'risk_level']

In [52]:
results["failure_reason"] = failed_payments.loc[
    results.index, "failure_reason"
]

results["payment_method"] = failed_payments.loc[
    results.index, "payment_method"
]

results["checkout_abandoned"] = failed_payments.loc[
    results.index, "checkout_abandoned"
]

In [53]:
results[
    [
        "amount",
        "failure_reason",
        "payment_method",
        "recovery_probability",
        "risk_level"
    ]
].head(10)

,amount,failure_reason,payment_method,recovery_probability,risk_level
790,708.49,bank_decline,UPI,0.265,low
31,2476.34,bank_decline,UPI,0.405,medium
519,1824.87,insufficient_funds,UPI,0.785,high
814,582.37,technical_timeout,UPI,0.625,medium
147,4364.14,bank_decline,UPI,0.745,high
928,1553.64,insufficient_funds,UPI,0.345,low
580,1054.45,bank_decline,UPI,0.330,low
347,1509.55,bank_decline,Netbanking,0.430,medium
382,1399.93,bank_decline,UPI,0.730,high
462,978.32,technical_timeout,Card,0.175,low


In [54]:
def agent_action(row):
    probability = row["recovery_probability"]
    reason = row["failure_reason"]

    if reason == "technical_timeout":
        return "retry"

    elif reason == "network_error":
        return "retry"

    elif reason == "authentication_failed":
        return "re_authenticate"

    elif reason == "insufficient_funds":
        return "reminder"

    elif reason == "bank_decline":
        if probability >= 0.70:
            return "alternate_payment"
        else:
            return "stop"

    else:
        return "review"

In [55]:
results["recommended_action"] = results.apply(
    agent_action,
    axis=1
)

In [56]:
results[
    [
        "amount",
        "failure_reason",
        "payment_method",
        "recovery_probability",
        "risk_level",
        "recommended_action"
    ]
].head(10)

,amount,failure_reason,payment_method,recovery_probability,risk_level,recommended_action
790,708.49,bank_decline,UPI,0.265,low,stop
31,2476.34,bank_decline,UPI,0.405,medium,stop
519,1824.87,insufficient_funds,UPI,0.785,high,reminder
814,582.37,technical_timeout,UPI,0.625,medium,retry
147,4364.14,bank_decline,UPI,0.745,high,alternate_payment
928,1553.64,insufficient_funds,UPI,0.345,low,reminder
580,1054.45,bank_decline,UPI,0.330,low,stop
347,1509.55,bank_decline,Netbanking,0.430,medium,stop
382,1399.93,bank_decline,UPI,0.730,high,alternate_payment
462,978.32,technical_timeout,Card,0.175,low,retry


In [57]:
results["recommended_action"].value_counts()

recommended_action
retry                28
reminder             22
stop                 15
re_authenticate       7
alternate_payment     6
Name: count, dtype: int64

testing 


In [58]:
 
import pandas as pd
new_payment = pd.DataFrame([{
    "amount": 2500,
    "failed_attempts": 2,
    "previous_successful_payments": 5,
    "days_since_last_payment": 30,
    "payment_method": "UPI",
    "failure_reason": "bank_decline",
    "checkout_abandoned": "no"
}])

In [59]:
new_payment = pd.get_dummies(
    new_payment,
    columns=["payment_method", "failure_reason", "checkout_abandoned"],
    drop_first=True,
    dtype=int
)

In [60]:
import os

for root, dirs, files in os.walk(".."):
    for file in files:
        if file.lower() == "payment.csv":
            print(os.path.abspath(os.path.join(root, file)))

In [61]:
import pandas as pd
import numpy as np

print("Pandas loaded")

Pandas loaded


In [62]:
import os

print(os.getcwd())
print(os.listdir("../data"))

e:\Ai Revenue\notebooks
['payments.csv']


In [63]:
df = pd.read_csv("../data/payments.csv")

print(df.shape)
print(df.head())

(1000, 13)
  payment_id customer_id   amount payment_method payment_status  \
0     P00001       C1103   409.59            UPI        success   
1     P00002       C1180  4086.34            UPI         failed   
2     P00003       C1093  1358.44         Wallet        success   
3     P00004       C1015  3745.69           Card        success   
4     P00005       C1107   447.73            UPI        success   

  failure_reason  failed_attempts  previous_successful_payments  \
0           none                0                             4   
1   bank_decline                3                             1   
2           none                0                             3   
3           none                0                             4   
4           none                0                             6   

   days_since_last_payment checkout_abandoned recovered recovery_action  \
0                      162                 no        no            none   
1                       75       

In [64]:
failed_payments = df[df["payment_status"] == "failed"].copy()

print(failed_payments.shape)

(389, 13)


In [65]:
%whos

Variable                 Type                      Data/Info
------------------------------------------------------------
LogisticRegression       type                      <class 'sklearn.linear_mo<...>stic.LogisticRegression'>
RandomForestClassifier   ABCMeta                   <class 'sklearn.ensemble.<...>.RandomForestClassifier'>
StandardScaler           type                      <class 'sklearn.preproces<...>ng._data.StandardScaler'>
X                        DataFrame                 Shape: (389, 12)
X_test                   DataFrame                 Shape: (78, 12)
X_test_scaled            ndarray                   78x12: 936 elems, type `float64`, 7488 bytes
X_train                  DataFrame                 Shape: (311, 12)
X_train_scaled           ndarray                   311x12: 3732 elems, type `float64`, 29856 bytes
accuracy                 float                     0.5897435897435898
accuracy_score           function                  <function accuracy_score at 0x00000187

In [66]:
ml_data = failed_payments[
    [
        "amount",
        "payment_method",
        "failure_reason",
        "failed_attempts",
        "previous_successful_payments",
        "days_since_last_payment",
        "checkout_abandoned",
        "recovered"
    ]
].copy()

ml_data["recovered"] = ml_data["recovered"].map({
    "no": 0,
    "yes": 1
})

ml_data = pd.get_dummies(
    ml_data,
    columns=["payment_method", "failure_reason", "checkout_abandoned"],
    drop_first=True,
    dtype=int
)

X = ml_data.drop("recovered", axis=1)
y = ml_data["recovered"]

print("X:", X.shape)
print("y:", y.shape)

X: (389, 12)
y: (389,)


In [67]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

model = RandomForestClassifier(
    n_estimators=200,
    random_state=42
)

model.fit(X_train, y_train)

print("Model trained successfully")

Model trained successfully


In [68]:
new_payment = new_payment.reindex(
    columns=X.columns,
    fill_value=0
)

print(new_payment.shape)
print(new_payment.columns.tolist())

(1, 12)
['amount', 'failed_attempts', 'previous_successful_payments', 'days_since_last_payment', 'payment_method_Netbanking', 'payment_method_UPI', 'payment_method_Wallet', 'failure_reason_bank_decline', 'failure_reason_insufficient_funds', 'failure_reason_network_error', 'failure_reason_technical_timeout', 'checkout_abandoned_yes']


In [69]:
recovery_probability = model.predict_proba(new_payment)[0][1]

print("Recovery Probability:", recovery_probability)

Recovery Probability: 0.73


In [70]:
if recovery_probability >= 0.70:
    risk_level = "high"
    recommended_action = "retry"
elif recovery_probability >= 0.40:
    risk_level = "medium"
    recommended_action = "reminder"
else:
    risk_level = "low"
    recommended_action = "stop"

print("Recovery Probability:", recovery_probability)
print("Risk Level:", risk_level)
print("Recommended Action:", recommended_action)

Recovery Probability: 0.73
Risk Level: high
Recommended Action: retry


In [71]:
import os

os.makedirs("../model", exist_ok=True)

joblib.dump(model, "../model/recovery_model.pkl")

print("Model saved successfully!")

NameError: name 'joblib' is not defined

In [ ]:
import mysql.connector

conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password="123456789",
    database="revenue_recovery"
)

print("MySQL connected successfully!")

MySQL connected successfully!


In [ ]:
cursor = conn.cursor()

cursor.execute("SELECT COUNT(*) FROM payments")

count = cursor.fetchone()[0]

print("Total payments:", count)

Total payments: 1000


In [ ]:
import pandas as pd
query = """
SELECT *
FROM payments
WHERE payment_status = 'failed'
"""

failed_data = pd.read_sql(query, conn)

print("Failed payments:", len(failed_data))
failed_data.head()

Failed payments: 389


C:\Users\DELL\AppData\Local\Temp\ipykernel_6636\160458400.py:8: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  failed_data = pd.read_sql(query, conn)


,payment_id,customer_id,amount,payment_method,payment_status,failure_reason,failed_attempts,previous_successful_payments,days_since_last_payment,checkout_abandoned,recovered,recovery_action,recovered_amount
0,P00002,C1180,4086.34,UPI,failed,bank_decline,3,1,75,no,no,stop,0.00
1,P00010,C1122,2361.70,UPI,failed,technical_timeout,4,2,169,no,yes,retry,2361.70
2,P00013,C1075,1620.07,Card,failed,insufficient_funds,4,4,109,no,no,escalate,0.00
3,P00016,C1117,636.84,UPI,failed,network_error,4,8,36,no,yes,reminder,636.84
4,P00020,C1131,844.41,Card,failed,authentication_failed,5,2,154,yes,no,escalate,0.00


In [ ]:
ml_input = failed_data[
    [
        "amount",
        "payment_method",
        "failure_reason",
        "failed_attempts",
        "previous_successful_payments",
        "days_since_last_payment",
        "checkout_abandoned"
    ]
].copy()

print("ML input shape:", ml_input.shape)

ML input shape: (389, 7)


In [ ]:
ml_input_encoded = pd.get_dummies(
    ml_input,
    columns=[
        "payment_method",
        "failure_reason",
        "checkout_abandoned"
    ],
    drop_first=True,
    dtype=int
)

print("Encoded shape:", ml_input_encoded.shape)
print(ml_input_encoded.columns.tolist())

Encoded shape: (389, 12)
['amount', 'failed_attempts', 'previous_successful_payments', 'days_since_last_payment', 'payment_method_Netbanking', 'payment_method_UPI', 'payment_method_Wallet', 'failure_reason_bank_decline', 'failure_reason_insufficient_funds', 'failure_reason_network_error', 'failure_reason_technical_timeout', 'checkout_abandoned_yes']


In [ ]:
feature_columns = [
    "amount",
    "failed_attempts",
    "previous_successful_payments",
    "days_since_last_payment",
    "payment_method_Netbanking",
    "payment_method_UPI",
    "payment_method_Wallet",
    "failure_reason_bank_decline",
    "failure_reason_insufficient_funds",
    "failure_reason_network_error",
    "failure_reason_technical_timeout",
    "checkout_abandoned_yes"
]

ml_input_encoded = ml_input_encoded.reindex(
    columns=feature_columns,
    fill_value=0
)

print("Final ML input shape:", ml_input_encoded.shape)

Final ML input shape: (389, 12)


In [ ]:
X = ml_input_encoded.copy()

y = failed_data["recovered"].map({
    "no": 0,
    "yes": 1
})

print("X shape:", X.shape)
print("y shape:", y.shape)
print("NaN in y:", y.isna().sum())

X shape: (389, 12)
y shape: (389,)
NaN in y: 0


In [ ]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(
    n_estimators=200,
    random_state=42
)

model.fit(X, y)

print("Random Forest trained successfully!")

Random Forest trained successfully!


In [ ]:
recovery_probabilities = model.predict_proba(ml_input_encoded)[:, 1]

print("Number of predictions:", len(recovery_probabilities))
print("First 10 probabilities:", recovery_probabilities[:10])

Number of predictions: 389
First 10 probabilities: [0.13  0.91  0.075 0.8   0.11  0.87  0.875 0.905 0.19  0.16 ]


In [ ]:
def get_risk_level(probability):
    if probability >= 0.70:
        return "high"
    elif probability >= 0.40:
        return "medium"
    else:
        return "low"

risk_levels = [get_risk_level(p) for p in recovery_probabilities]

print("High:", risk_levels.count("high"))
print("Medium:", risk_levels.count("medium"))
print("Low:", risk_levels.count("low"))

High: 180
Medium: 6
Low: 203


In [ ]:
def get_recommended_action(reason, probability):
    if reason == "technical_timeout":
        return "retry"

    elif reason == "network_error":
        return "retry"

    elif reason == "authentication_failed":
        return "re_authenticate"

    elif reason == "insufficient_funds":
        return "reminder"

    elif reason == "bank_decline":
        if probability >= 0.70:
            return "alternate_payment"
        else:
            return "stop"

    else:
        return "review"


recommended_actions = [
    get_recommended_action(reason, probability)
    for reason, probability in zip(
        failed_data["failure_reason"],
        recovery_probabilities
    )
]

print("Number of actions:", len(recommended_actions))
print(pd.Series(recommended_actions).value_counts())

Number of actions: 389
retry                131
reminder              94
stop                  61
alternate_payment     52
re_authenticate       51
Name: count, dtype: int64


In [ ]:
results_to_save = pd.DataFrame({
    "payment_id": failed_data["payment_id"].values,
    "recovery_probability": recovery_probabilities,
    "risk_level": risk_levels,
    "recommended_action": recommended_actions,
    "failure_reason": failed_data["failure_reason"].values
})

print(results_to_save.shape)
results_to_save.head()

(389, 5)


,payment_id,recovery_probability,risk_level,recommended_action,failure_reason
0,P00002,0.130,low,stop,bank_decline
1,P00010,0.910,high,retry,technical_timeout
2,P00013,0.075,low,reminder,insufficient_funds
3,P00016,0.800,high,retry,network_error
4,P00020,0.110,low,re_authenticate,authentication_failed


In [ ]:
insert_query = """
INSERT INTO recovery_results (
    payment_id,
    recovery_probability,
    risk_level,
    recommended_action,
    action_status,
    recovered,
    recovered_amount,
    failure_reason
)
VALUES (%s, %s, %s, %s, %s, %s, %s, %s)
"""

rows = [
    (
        payment_id,
        float(probability),
        risk,
        action,
        "recommended",
        recovered,
        float(recovered_amount),
        reason
    )
    for payment_id, probability, risk, action, recovered, recovered_amount, reason
    in zip(
        failed_data["payment_id"],
        recovery_probabilities,
        risk_levels,
        recommended_actions,
        failed_data["recovered"],
        failed_data["recovered_amount"],
        failed_data["failure_reason"]
    )
]

cursor.executemany(insert_query, rows)
conn.commit()

print("Inserted rows:", cursor.rowcount)

Inserted rows: 389


In [ ]:
import joblib

loaded_model = joblib.load("../model/recovery_model.pkl")

print("Model loaded successfully!")
print(type(loaded_model))

Model loaded successfully!
<class 'sklearn.ensemble._forest.RandomForestClassifier'>


In [ ]:
model = loaded_model

print("Model ready:", type(model))

Model ready: <class 'sklearn.ensemble._forest.RandomForestClassifier'>


In [ ]:
import os
import joblib

os.makedirs("../model", exist_ok=True)

model_path = "../model/recovery_model.pkl"

joblib.dump(model, model_path)

print("Saved:", os.path.abspath(model_path))
print("File exists:", os.path.exists(model_path))

Saved: e:\Ai Revenue\model\recovery_model.pkl
File exists: True


In [ ]:
import os
import joblib

os.makedirs("../model", exist_ok=True)

model_path = "../model/recovery_model.pkl"

joblib.dump(model, model_path)

print("Saved:", os.path.abspath(model_path))
print("File exists:", os.path.exists(model_path))

Saved: e:\Ai Revenue\model\recovery_model.pkl
File exists: True
